# H2L1 — Learning the angle

**Week 6, notebook 2.** In the slides every angle of RoPE was set by hand. In
this notebook gradient descent chooses the angles, in two parts:

1. **One pair, one query.** The query has two numbers, and the loss only says
   "put the attention on the token three positions back"; it never mentions
   angles. The angle (where to look) converges in about 20 steps, and the
   length (how strongly to attend) keeps growing afterwards.
2. **A trained head.** A small RoPE transformer is trained on sentences such
   as *"the capital of france is paris"* in three templates, so the country is
   at a different distance from "is" in each. One head learns to attend to
   **"france"** when the model must predict **"paris"**, and follows the country
   when it is changed.

**Colab:** Runtime → Run all. About a minute on the CPU runtime.


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)   # a NumPy/torch version notice, not an error

torch.manual_seed(0); np.random.seed(0)
DEV = "cuda" if torch.cuda.is_available() else "cpu"

C_BLUE, C_RED, C_TEAL, C_PURPLE, C_GRAY = \
    "#1F4E79", "#C00000", "#0E7C7B", "#7030A0", "#595959"


# Part 1 of 2 — One pair and one query

The example of the slides, made trainable: $d_{head}=2$ (one pair), $\theta = 1$
rad/position, six keys all starting $[1,0]$, the query sitting at position
5. The query's two numbers are the **only parameters**. The loss is
cross-entropy toward key 2 — i.e. "attend 3 back". No angle appears
anywhere in the objective.


In [ ]:
TH, M_Q, TARGET = 1.0, 5, 2                     # want attention on pos 2 = 3 back

def rot(v, ang):
    c, s = math.cos(ang), math.sin(ang)
    return torch.stack([v[0] * c - v[1] * s, v[0] * s + v[1] * c])

keys = torch.stack([rot(torch.tensor([1.0, 0.0]), n * TH) for n in range(6)])
qp = torch.randn(2, generator=torch.Generator().manual_seed(3)).requires_grad_(True)
opt = torch.optim.Adam([qp], lr=0.15)

hist = {"q": [], "phi": [], "r": [], "P": []}
for step in range(121):
    scores = (rot(qp, M_Q * TH) * keys).sum(-1) * 4.0
    loss = F.cross_entropy(scores[None], torch.tensor([TARGET]))
    hist["q"].append(qp.detach().clone().numpy())
    hist["phi"].append(math.atan2(qp[1].item(), qp[0].item()))
    hist["r"].append(qp.norm().item())
    hist["P"].append(scores.softmax(-1).detach().numpy())
    opt.zero_grad(); loss.backward(); opt.step()

print(f"{'step':>5} {'query [a,b]':>20} {'phi_q':>8} {'-phi/theta':>11} "
      f"{'P(3 back)':>10} {'length':>8}")
for s in [0, 10, 20, 40, 80, 120]:
    q = hist["q"][s]
    print(f"{s:>5} [{q[0]:+.3f},{q[1]:+.3f}]{'':>4} {hist['phi'][s]:>+8.3f} "
          f"{-hist['phi'][s]/TH:>11.2f} {hist['P'][s][TARGET]:>10.4f} "
          f"{hist['r'][s]:>8.3f}")


Read the columns: `-phi/theta` — the *distance the angle asks for* — hits
3.0 around step 20 and freezes. The length is still climbing at step 120.
Once the head knows **where** to look, the only way to improve the loss is
to say it **louder**: bigger scores, sharper softmax.

Three views of the same 121 steps:


In [ ]:
fig = plt.figure(figsize=(13, 4.0))

# (1) the query's trajectory in its plane: turn first, then lengthen
ax = fig.add_subplot(1, 3, 1)
Q = np.array(hist["q"])
sc = ax.scatter(Q[:, 0], Q[:, 1], c=np.arange(len(Q)), cmap="viridis", s=14)
ax.plot(Q[:, 0], Q[:, 1], color=C_GRAY, lw=0.5, alpha=0.5)
ax.annotate("start", Q[0], fontsize=9, color=C_GRAY,
            xytext=(6, 6), textcoords="offset points")
ax.annotate("step 120", Q[-1], fontsize=9, color=C_GRAY,
            xytext=(6, -10), textcoords="offset points")
phi_star = math.atan2(Q[-1, 1], Q[-1, 0])
ax.plot([0, 3 * math.cos(phi_star)], [0, 3 * math.sin(phi_star)],
        ls=":", color=C_RED, lw=1)
ax.text(1.1 * math.cos(phi_star) - 1.6, 1.1 * math.sin(phi_star) - 0.35,
        "the learned direction:\n'3 tokens back'", fontsize=8, color=C_RED)
ax.set_aspect("equal"); ax.set_title("(1) the query over training", fontsize=10)
ax.axhline(0, color=C_GRAY, lw=0.4), ax.axvline(0, color=C_GRAY, lw=0.4)
plt.colorbar(sc, ax=ax, fraction=0.046, label="step")

# (2) the two degrees of freedom on their own clocks
ax = fig.add_subplot(1, 3, 2)
ax.plot(np.array(hist["phi"]) / -TH, color=C_BLUE, lw=1.6,
        label=r"angle as distance  $-\varphi_q/\theta$")
ax.axhline(3, color=C_BLUE, ls=":", lw=1)
ax.set_ylabel("requested distance", color=C_BLUE)
ax.set_xlabel("step")
ax2 = ax.twinx()
ax2.plot(hist["r"], color=C_RED, lw=1.6, label="length")
ax2.set_ylabel(r"length $\|q\|$", color=C_RED)
ax.axvline(20, color=C_GRAY, ls=":", lw=1)
ax.text(22, 1.0, "angle settled by step 20;\nlength keeps growing", fontsize=8,
        color=C_GRAY)
ax.set_title("(2) selected distance (blue) and length (red)", fontsize=10)

# (3) the attention row over training: mass slides onto D=3 and sharpens
ax = fig.add_subplot(1, 3, 3)
P = np.array(hist["P"])                                # [steps, 6]
im = ax.imshow(P.T, aspect="auto", cmap="Blues", vmin=0, vmax=1,
               interpolation="nearest")
ax.set_yticks(range(6))
ax.set_yticklabels([f"pos {n} ({M_Q - n} back)" for n in range(6)], fontsize=8)
ax.axhline(TARGET, color=C_RED, lw=1, ls="--")
ax.set_xlabel("step"), ax.set_title("(3) the attention row, every step", fontsize=10)
plt.colorbar(im, ax=ax, fraction=0.046, label="attention")
plt.tight_layout(); plt.show()


In a real model the query is not the parameter — $W_Q$ is, and the gradient
flows one step further back: $W_Q$ learns a **function from token content
to angle**. Different token in, different distance requested. That is what
Part B now shows happening.

# Part 2 of 2 — A trained head

### The setup: "the capital of france is ___"

A tiny word-level transformer (2 layers, 4 heads, RoPE — the same
`rope_rotate` as H3L1) trained on capital facts in **three sentence
shapes**, so the country sits at a *different distance* from "is" depending
on the sentence:

```
the capital of france is paris
the capital of france , i think , is paris
france 's capital is paris
```

A fixed-offset head cannot solve all three. To say "paris" the model must
*find the country by content*, wherever it sits — attention, doing the only
thing it does.


In [ ]:
FACTS = [("france", "paris"), ("japan", "tokyo"), ("italy", "rome"),
         ("spain", "madrid"), ("egypt", "cairo"), ("chile", "santiago"),
         ("kenya", "nairobi"), ("norway", "oslo")]
TEMPLATES = ["the capital of {c} is {k}",
             "the capital of {c} , i think , is {k}",
             "{c} 's capital is {k}"]

words = sorted({w for t in TEMPLATES for w in t.split() if "{" not in w}
               | {w for f in FACTS for w in f})
vocab = {w: i for i, w in enumerate(["<pad>"] + words)}
ivocab = {i: w for w, i in vocab.items()}
V = len(vocab)
enc = lambda s: torch.tensor([vocab[w] for w in s.split()])
print(f"vocabulary: {V} words")

D_MODEL, H = 64, 4
D_HEAD = D_MODEL // H

def rope_rotate(x, positions):
    j = torch.arange(D_HEAD // 2, device=x.device, dtype=torch.float32)
    theta = 10000.0 ** (-2.0 * j / D_HEAD)
    ang = positions.float()[:, None] * theta[None, :]
    c, s = ang.cos()[None, None], ang.sin()[None, None]
    x1, x2 = x[..., 0::2], x[..., 1::2]
    out = torch.empty_like(x)
    out[..., 0::2] = x1 * c - x2 * s
    out[..., 1::2] = x1 * s + x2 * c
    return out

class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(D_MODEL), nn.LayerNorm(D_MODEL)
        self.wq, self.wk, self.wv, self.wo = (nn.Linear(D_MODEL, D_MODEL, bias=False)
                                              for _ in range(4))
        self.mlp = nn.Sequential(nn.Linear(D_MODEL, 4 * D_MODEL), nn.GELU(),
                                 nn.Linear(4 * D_MODEL, D_MODEL))
        self.last_attn = None
    def forward(self, x):
        Bn, Tn, _ = x.shape
        h = self.ln1(x)
        split = lambda t: t.view(Bn, Tn, H, D_HEAD).transpose(1, 2)
        pos = torch.arange(Tn, device=x.device)
        q = rope_rotate(split(self.wq(h)), pos)
        k = rope_rotate(split(self.wk(h)), pos)
        v = split(self.wv(h))
        s = q @ k.transpose(-1, -2) / math.sqrt(D_HEAD)
        s = s + torch.triu(torch.full((Tn, Tn), -torch.inf, device=x.device), 1)
        p = F.softmax(s, -1)
        self.last_attn = p.detach()
        x = x + self.wo((p @ v).transpose(1, 2).reshape(Bn, Tn, D_MODEL))
        return x + self.mlp(self.ln2(x))

class LM(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(V, D_MODEL)
        self.blocks = nn.ModuleList([Block(), Block()])
        self.ln_f = nn.LayerNorm(D_MODEL)
        self.head = nn.Linear(D_MODEL, V)
    def forward(self, x):
        h = self.emb(x)
        for b in self.blocks:
            h = b(h)
        return self.head(self.ln_f(h))

def sample_batch(n=32):
    rows = []
    g = torch.randint(0, len(FACTS), (n,)), torch.randint(0, len(TEMPLATES), (n,))
    for fi, ti in zip(*g):
        c, k = FACTS[fi]
        rows.append(enc(TEMPLATES[ti].format(c=c, k=k)))
    L = max(len(r) for r in rows)
    x = torch.zeros(n, L, dtype=torch.long)
    for i, r in enumerate(rows):
        x[i, :len(r)] = r
    return x.to(DEV)

PROBE = "the capital of france is"
probe_ids = enc(PROBE)[None].to(DEV)
probe_toks = PROBE.split()
FR, PARIS = probe_toks.index("france"), vocab["paris"]


Train, taking a snapshot every few steps: the model's P("paris") after the
probe, and every head's attention from "is" back over the sentence.


In [ ]:
model = LM().to(DEV)
opt = torch.optim.Adam(model.parameters(), lr=2e-3)
SNAPS = [0, 10, 20, 40, 80, 150, 250, 400]
snap_attn, snap_p, losses = [], [], []

for step in range(401):
    if step in SNAPS:
        with torch.no_grad():
            logits = model(probe_ids)
            p_paris = F.softmax(logits[0, -1], -1)[PARIS].item()
            # [layer, head, key] attention of the final "is" query
            a = torch.stack([b.last_attn[0, :, -1] for b in model.blocks])
        snap_attn.append(a.cpu()); snap_p.append(p_paris)
    x = sample_batch()
    tgt = x[:, 1:].reshape(-1)
    loss = F.cross_entropy(model(x)[:, :-1].reshape(-1, V), tgt,
                           ignore_index=0)
    opt.zero_grad(); loss.backward(); opt.step()
    losses.append(loss.item())

# the head that ends up doing the work: max final attention onto "france"
flat = snap_attn[-1][:, :, FR]
L_STAR, H_STAR = np.unravel_index(flat.argmax(), flat.shape)
print(f"the working head: layer {L_STAR}, head {H_STAR}")
print(f"P('paris' | '{PROBE}') over training: " +
      "  ".join(f"{s}:{p:.3f}" for s, p in zip(SNAPS, snap_p)))


### Watch the head learn to point

Left: the chosen head's attention from **"is"**, one row per snapshot —
at step 0 the mass is wherever initialization left it; a few dozen steps
in, it has found **"france"** and commits. Right: P("paris") rising as the
routing locks in — the same two-phase story as Part A, now with words.


In [ ]:
A = torch.stack([a[L_STAR, H_STAR] for a in snap_attn]).numpy()  # [snaps, T]

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.9),
                             gridspec_kw={"width_ratios": [1.2, 1]})
im = a1.imshow(A, aspect="auto", cmap="Blues", vmin=0, vmax=1)
a1.set_xticks(range(len(probe_toks))), a1.set_xticklabels(probe_toks)
a1.set_yticks(range(len(SNAPS)))
a1.set_yticklabels([f"step {s}" for s in SNAPS], fontsize=8)
a1.axvline(FR, color=C_RED, ls="--", lw=1)
for i in range(A.shape[0]):
    jmax = A[i].argmax()
    a1.text(jmax, i, f"{A[i, jmax]:.2f}", ha="center", va="center", fontsize=7.5,
            color="white" if A[i, jmax] > 0.55 else "#222222")
a1.set_title(f'where "is" looks (layer {L_STAR}, head {H_STAR}) — training '
             "downward", fontsize=10)
plt.colorbar(im, ax=a1, fraction=0.03)

a2.plot(SNAPS, snap_p, "o-", color=C_TEAL, lw=1.8)
a2.set_xlabel("step"), a2.set_ylabel('P("paris")'), a2.set_ylim(-0.04, 1.04)
a2.set_title("probability of the answer", fontsize=10)
a2.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()


### And it follows the country, not the slot

Same trained weights, same sentence shape, different country — including a
template where the country sits somewhere else entirely. The head follows
the **country**, and the prediction follows the head.


In [ ]:
tests = ["the capital of france is", "the capital of japan is",
         "the capital of egypt , i think , is", "norway 's capital is"]
answers = ["paris", "tokyo", "cairo", "oslo"]

fig, axes = plt.subplots(len(tests), 1, figsize=(10, 1.5 * len(tests)),
                         sharex=False)
for ax, snt, ans in zip(axes, tests, answers):
    ids = enc(snt)[None].to(DEV)
    toks = snt.split()
    with torch.no_grad():
        logits = model(ids)
        p = F.softmax(logits[0, -1], -1)
    a = model.blocks[L_STAR].last_attn[0, H_STAR, -1].cpu().numpy()
    top = p.argmax().item()
    bars = ax.bar(range(len(toks)), a, color=C_BLUE)
    ci = toks.index([c for c, _ in FACTS if c in toks][0])
    bars[ci].set_color(C_RED)
    ax.set_xticks(range(len(toks))), ax.set_xticklabels(toks, fontsize=9)
    ok = "OK" if ivocab[top] == ans else "WRONG"
    print(f"{snt:38s} -> {ivocab[top]:10s} P={p[top]:.2f}  [{ok}]")
    ax.set_ylabel("attn", fontsize=8)
    ax.set_title(f'→ predicts "{ivocab[top]}" (P={p[top]:.2f})  [{ok}]',
                 fontsize=9, loc="right", y=0.65)
    ax.spines[["top", "right"]].set_visible(False)
fig.suptitle('the same head looks at the country in each sentence', fontsize=11)
plt.tight_layout(); plt.show()


## What H2L1 showed

| | Part A (one pair) | Part B (real sentence) |
|---|---|---|
| the objective mentions | a target position only | the next word only |
| what the gradient built | an **angle** = a distance request | a head that finds "france" by content |
| order of learning | angle first (~20 steps), length forever after | routing first, confidence after |
| nothing hand-set | $\varphi_q$ found by descent | which head does it, found by descent |

The two experiments are the same fact at two scales: **RoPE gives the
gradient a smooth dial from content to *where to look*, and training turns
it until the loss drops.** No example in training ever labeled an
attention target — the routing is a side effect of predicting the next
word.

**Next (H3L1):** what happens beyond the training length, for learned
absolute positions and for RoPE, and how much PI recovers.
